In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import List, Dict

# tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507", local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct", local_files_only=True)
tokenizer.pad_token_id = tokenizer.eos_token_id


In [ ]:
def tokenize(
    tokenizer: AutoTokenizer, 
    text: str, 
    role: str, 
    max_length: int | None = None,
    keep_bos: bool = False,
    add_generation_prompt: bool = False,
):
    if not keep_bos and tokenizer.bos_token is not None and max_length is not None:
        # the bos token is not counted in max_length
        max_length = max_length + 1
    input_ids = tokenizer.apply_chat_template(
        [{"role": role, "content": text}], tokenize=True, 
        max_length=max_length, truncation=max_length is not None,
        add_generation_prompt=add_generation_prompt,
    )
    if not keep_bos and input_ids[0] == tokenizer.bos_token_id:
        input_ids = input_ids[1:]
    return input_ids

# 高德数据集

In [ ]:
import datasets
import string
from pprint import pprint

data = datasets.load_dataset("/mnt/nas1/duchuheng/datasets/longmagpie", split="train")

def extract_document_magpie(sample):
    content = sample['messages'][0]['content']
    answer = sample['messages'][1]['content'].strip()
    content_sentences = content.split('.')
    content, question = '.'.join(content_sentences[:-1]), content_sentences[-1]
    documents = []
    document = ''
    for line in content.split('\n'):
        line = line.strip()
        if not line:
            continue
        if line == "===Document Separator===":
            if document:
                documents.append(document)
            document = ''
            continue
        if len(document) + len(line) > 6400:
            documents.append(document)
            document = ''
        else:
            document += line + '\n'
    if document:
        documents.append(document)
    if len(documents) > 10:
        return {'documents': None, 'question': None, 'answer': [None]}
    return {'documents': documents, 'question': question.strip(), 'answer': [answer]}

data = data.map(extract_document_magpie, remove_columns=data.column_names, batched=False, num_proc=32).filter(lambda x: x['documents'] is not None, num_proc=64)

print(len(data))
amapdata = data.select(range(10000))

In [ ]:
data.save_to_disk('/mnt/nas1/duchuheng/datasets/longmagpie_1536')

In [ ]:
import os
import datasets
import string
from pprint import pprint
from typing import Any, Dict

data = datasets.load_dataset("Yukang/LongAlpaca-16k-length", split="train")

CONTEXT_BEGIN: str = "The paper begins. "
CONTEXT_END: str = "Now the paper ends."

def _preprocess_sample(sample: Dict[str, Any]) -> Dict[str, Any]:
    documents = []
    last_document = ''
    if CONTEXT_BEGIN not in sample['instruction'] or CONTEXT_END not in sample['instruction']:
        return {'documents': None, 'question': None, 'answer': None}
    context = sample['instruction'].split(CONTEXT_BEGIN, 1)[1]
    context, question = context.split(CONTEXT_END, 1)
    if len(question) > 1024:
        return {'documents': None, 'question': None, 'answer': None}
    for line in context.split('\n'):
        if (len(line) > 0 and line[0] in string.digits) or (len(line) + len(last_document) > 6000):
            if len(last_document) > 0:
                documents.append(last_document)
            last_document = ''
        last_document += line + '\n'
    if len(last_document) > 0:
        documents.append(last_document)
    documents = documents[:10]
    return {
        'question': question.strip(),
        'documents': documents,
        'answer': [sample['output'].strip()],
    }

data = data.map(_preprocess_sample, remove_columns=data.column_names, batched=False, num_proc=64).filter(lambda x: x['documents'] is not None, num_proc=64)

print(len(data))
amapdata = data

In [ ]:
data.save_to_disk('/mnt/nas1/duchuheng/datasets/longalpaca_processed_long')

# 多文档测试集

In [ ]:
import os
import numpy as np
from collections import Counter
from python.inference.mdocdataset import load_mdoc_dataset
from typing import Any, Dict
import matplotlib.pyplot as plt

# amapdata = load_mdoc_dataset(
#     "amap", 
#     "../datasets/AmapData.csv",
#     load_full=True
# )
# amapdata = load_mdoc_dataset(
#     "musique", 
#     "../datasets/musique_ans_v1.0_train.jsonl",
# )
amapdata = load_mdoc_dataset("wikimqa")
# amapdata = load_mdoc_dataset("longalpaca", "../datasets/longalpaca_prompts.txt")


In [ ]:
for sample in amapdata:
    if sample['qid'] == '41ac2a4beb0af8f58d01863a62b90692f7c7d74b5e3a58d9':
        break

with open('./saved_kv/sample.txt', 'w') as f:
    # f.write(sample['system_prompt'] + '\n========\n'.join(sample['documents']) + sample['question'] + sample['answer'][0])
    f.write('\n# #\n'.join(sample['documents']) + "\n========\n" + sample['question']  + "\n========\n" + sample['answer'][0])

In [ ]:
from tqdm import tqdm
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507", local_files_only=True)
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct", local_files_only=True)
context_lengths = []

for sample in tqdm(amapdata):
    context_lengths.extend(map(len, tokenizer(sample['documents']).input_ids))

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

# plot context_lengths as histogram

# plt.hist([x for x in context_lengths if x < 2048], bins=50)
plt.hist(context_lengths, bins=50)
plt.show()
print(np.mean(context_lengths), np.std(context_lengths), min(context_lengths), max(context_lengths))

In [ ]:
def plot_hist_and_stat(doc_lengths, num_bins, title, xlabel):
    # 绘制直方图
    plt.figure(figsize=(6, 2))
    plt.hist(doc_lengths, bins=num_bins, alpha=0.7, color='steelblue')
    plt.xlabel(xlabel)
    plt.ylabel('Frequency')
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.show()

    max_doc_length = max(doc_lengths)
    max_doc_index = doc_lengths.index(max_doc_length)

    # 打印基本统计信息
    print(f"Mean: {np.mean(doc_lengths):.2f}")
    print(f"Std: {np.std(doc_lengths):.2f}")
    print(f"Min: {min(doc_lengths)}")
    print(f"Max: {max(doc_lengths)}")

def cut_long_documents(docuemnt: str):
    passages = [passage for passage in docuemnt.split('\n\n') if passage.strip()]
    docuemnts = []
    last_passage = ''
    for passage in passages:
        if len(last_passage) + len(passage) > 2048:
            docuemnts.append(last_passage)
            last_passage = passage
        else:
            last_passage += passage + '\n\n'
    if last_passage:
        docuemnts.append(last_passage)
    return docuemnts

# 获取所有sample的documents数量
# doc_lengths = [len(sample['documents']) for sample in amapdata]
doc_lengths = []
for sample in amapdata:
    # doc_lengths.extend(map(len, sample['documents']))
    doc_lengths.append(sum(map(len, (cut_long_documents(doc) for doc in sample['documents']))))
plot_hist_and_stat(doc_lengths, max(doc_lengths), "Number of Documents per Sample", "Number of Documents")


# 获取所有sample的documents长度
doc_lengths = []
for sample in amapdata:
    doc_lengths.extend(map(len, sample['documents']))
    # for doc in sample['documents']:
        # doc_lengths.extend(len(passage) for passage in cut_long_documents(doc))
plot_hist_and_stat(doc_lengths, 50, "Distribution of Documents Length", "Length of Documents")

# 获取所有sample长度
# doc_lengths = [
#     len(sample['system_prompt']) + sum(map(len, sample['documents'])) + len(sample['question'])
#     # sum(map(len, sample['documents'])) + len(sample['question'])
#     for sample in amapdata
# ]
# plot_hist_and_stat(doc_lengths, 50, "Distribution of Sample Length", "Length of Sample")

long_samples = []
for sample in amapdata:
    if any(map(lambda doc: len(doc) > 8192, sample['documents'])):
        long_samples.append(sample)

print("#request containing long documents: ", len(long_samples), f", which is {len(long_samples) / len(amapdata) * 100:.2f}%")


In [ ]:
# for i, sample in enumerate(amapdata):
#     if sample['qid'] == '21474b8617603387413106658d04e6':
#         max_doc_index = i
with open('saved_kv/sample.txt', 'w') as f:
    sample = amapdata[0]
    f.write(sample['qid'] + '\n\n')
    f.write(sample['system_prompt'])
    f.write('\n'.join(sample['documents']))
    f.write(sample['question'])
    f.write(sample['answer'][0])

In [ ]:
def get_reusable(dataset):
    doc_dict = {}
    for sample in dataset:
        for doc in sample['documents']:
            doc_dict[doc] = doc_dict.get(doc, 0) + 1
    return doc_dict, sum(doc_dict.values())

doc_dict, doc_num = get_reusable(amapdata)
doc_resuable = sum(v - 1 for v in doc_dict.values())
unique_prompts = len(set(sample['system_prompt'] for sample in amapdata))
print("Unique Prompts:", unique_prompts)
print("Total Documents:", doc_num)
print("Unique Documents:", len(doc_dict))
print("Reusable Documents:", doc_resuable)
print("Reusable Rate:", round(doc_resuable / doc_num, 4))

resuable_freq = [(k, v) for k, v in Counter(doc_dict.values()).items()]
# sort by key
resuable_freq.sort(key=lambda x: x[0])
print(resuable_freq)

In [ ]:
for k, v in doc_dict.items():
    if v == 3:
        print(len(k))
        print(k)

# LLM Judge

In [ ]:
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt

judge_scores = np.load("results/amap/llm_judge/score.npz")
print(judge_scores.files)

In [ ]:
def plot_multiple_rating_distributions(ratings_arrays, labels=None, title="Rating Distribution", save_path=None, figsize=(5, 3)):
    if isinstance(ratings_arrays, np.ndarray):
        ratings_arrays = [ratings_arrays]
    if labels is None:
        labels = [f'Dataset {i+1}' for i in range(len(ratings_arrays))]
    fig, ax = plt.subplots(figsize=figsize)
    colors = ['skyblue', 'lightcoral', 'lightgreen', 'gold', 'plum', 'orange', 'lightpink']
    for i, (ratings, label) in enumerate(zip(ratings_arrays, labels)):
        if len(ratings) == 0:
            continue
        rating_counts = Counter(ratings)
        all_ratings = [1, 2, 3, 4, 5]
        counts = [rating_counts.get(r, 0) for r in all_ratings]
        percentages = [count / len(ratings) * 100 if len(ratings) > 0 else 0 for count in counts]
        x_pos = [r + (i - len(ratings_arrays)/2 + 0.5) * 0.2 for r in all_ratings]
        ax.bar(x_pos, percentages, width=0.2, alpha=0.7, label=label, 
               color=colors[i % len(colors)])
    ax.set_xlabel('Rating')
    ax.set_ylabel('Percentage (%)')
    ax.set_title(title)
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_xticklabels(['1', '2', '3', '4', '5'])
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


plot_labels = ['fullreuse', 'epic16', 'recompute', 'cacheblend']
plot_arrays = [judge_scores[label] for label in plot_labels]
plot_multiple_rating_distributions(plot_arrays, plot_labels)
# print statistics
for label, array in zip(plot_labels, plot_arrays):
    print(f"{label}: {np.mean(array):.6f}")

# 画 Gist Token 实验结果

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import re
import pandas as pd
from pathlib import Path

def plot_gist_results(dataset: str, save_prefix: str = "gist", model_name: str = "qwen3-4b"):
    model = model_name
    # 读取实验结果json文件
    # with open(f'results/gist/{model_name}/evaluate_checkpoints_{dataset}.jsonl', 'r') as f:  # 请将'experiment_results.json'替换为你的文件路径
    with open(f'results/gist/{model_name}.jsonl', 'r') as f:
        results = [json.loads(line) for line in f]
        if results and 'dataset' in results[0]:
            results = [result for result in results if result['dataset'] == dataset]

    def get_baseline(dataset, method):
        with open(f'results/{dataset}/{model_name}_{method}_summary.json', 'r') as f: 
            results = json.load(f)
        if "f1_score" in results:
            return round(results["f1_score"], 6)
        return round(results["exact_match"], 6)

    # 定义baseline参考线数值（请根据实际情况修改）
    methods = ["reuse", "epic32", "fr", "blend"]
    if model_name == "llama3.1-8b":
        methods = ["reuse", "epic32", "fr", "blend", "blockattn"]
    baselines = {method: get_baseline(dataset, method) for method in methods}

    # 提取模型根目录并组织数据
    model_data = {}

    for result in results:
        model_path = result["model"]
        # 提取根目录模型名称（例如：qwen3-4b-inst/compressed_sft-16x）
        path_parts = model_path.split('/')
        if len(path_parts) >= 2:
            # 获取倒数第二层和最后一层作为模型标识
            root_model = f"{path_parts[-3]}/{path_parts[-2]}" if len(path_parts) >= 3 else f"{path_parts[-2]}"
            
            if root_model not in model_data:
                model_data[root_model] = []
            
            # 从model路径中提取checkpoint步数
            checkpoint_match = re.search(r'checkpoint-(\d+)', model_path)
            if checkpoint_match:
                steps = int(checkpoint_match.group(1))
                model_data[root_model].append({
                    'steps': steps,
                    'score': result['score'],
                    'model': model_path
                })

    model_names_sorted = sorted(model_data.keys())
    # print('\n'.join(model_names_sorted))
    model_avg_scores = []

    # 对每个模型的数据按steps排序，并计算最大step
    for model_name in model_names_sorted:
        model_data[model_name].sort(key=lambda x: x['steps'])
        # 计算相对进度（当前step / 最大step）
        if model_data[model_name]:  # 如果有数据
            max_step = max(item['steps'] for item in model_data[model_name])
            for item in model_data[model_name]:
                item['relative_progress'] = item['steps'] / max_step
        # 计算平均分
        model_avg_scores.append((
            model_name, 
            round(np.mean([item['score'] for item in model_data[model_name]]), 6)
        ))

    # only plot the 10 best models
    model_avg_scores.sort(key=lambda x: x[1], reverse=True)
    plot_model_names = [name for name, _ in model_avg_scores[:10]]

    # 绘制图表
    plt.figure(figsize=(7, 6))

    # 为每个模型绘制一条线
    colors = plt.cm.tab10(range(len(model_data)))  # 使用不同的颜色
    markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']  # 不同的标记

    for idx, model_name in enumerate(plot_model_names):
        data_list = model_data[model_name]
        if len(data_list) > 0:  # 只绘制有数据的模型
            df = pd.DataFrame(data_list)
            color = colors[idx % len(colors)]
            marker = markers[idx % len(markers)]
            
            plt.plot(df['relative_progress'], df['score'], 
                    marker=marker, linewidth=2, markersize=8, 
                    label=model_name, color=color)
            
            # 在图上显示具体的数值点
            for i, row in df.iterrows():
                plt.annotate(f'{row["score"]:.3f}', 
                            (row['relative_progress'], row['score']),
                            textcoords="offset points", 
                            xytext=(0,10), 
                            ha='center', 
                            fontsize=8,
                            color=color)

    # 添加baseline参考线
    baseline_colors = plt.cm.tab10(range(len(baselines)))
    for idx, (method, baseline_value) in enumerate(baselines.items()):
        plt.axhline(y=baseline_value, color=baseline_colors[idx % len(baseline_colors)], linestyle='--', linewidth=2, 
                label=f'{method} ({baseline_value})')

    plt.xlabel('Training Progress (current step / max step)', fontsize=12)
    plt.ylabel('Test Score', fontsize=12)
    plt.title(dataset, fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.legend(bbox_to_anchor=(1.8, 1.05), loc='upper right', fontsize=10)

    # 设置x轴为0-1范围
    plt.xlim(0.15, 1.05)
    plt.xticks([0.2, 0.4, 0.6, 0.8, 1.0])

    # 设置图表样式
    # plt.tight_layout()
    figure_dir = f'figures/{model}/'
    Path(figure_dir).mkdir(parents=True, exist_ok=True)
    plt.savefig(f'{figure_dir}/{save_prefix}_{dataset}.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
model = "qwen3-4b"
# model = "llama3.1-8b"
# model = "qwen3-8b"
plot_gist_results("hotpotqa", model_name=model)
plot_gist_results("musique", model_name=model)
plot_gist_results("wikimqa", model_name=model)
plot_gist_results("samsum", model_name=model)
plot_gist_results("multinews", model_name=model)

# KV Cache DIFF vs. Recompute Ratio


In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import glob

def load_data(model_name, head_idx):
    """加载数据并计算每层的平均差异比"""
    base_path = f"./saved_kv/kv_diff/{model_name}/{head_idx}"
    
    # 获取所有jsonl文件
    files = glob.glob(f"{base_path}/*.jsonl")
    
    data_dict = {}
    
    for file_path in files:
        filename = os.path.basename(file_path).replace('.jsonl', '')
        
        key_differences = []
        value_differences = []
        
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line.strip())
                key_differences.append(np.mean(data['key_differences']))
                value_differences.append(np.mean(data['value_differences']))
        
        # 计算每层的平均差异比
        if key_differences:
            
            if filename == 'none':
                ratio = 0
            else:
                # 提取重算百分比
                ratio = int(filename.split('-')[1])
            
            data_dict[ratio] = {
                'key': key_differences,
                'value': value_differences,
            }
    
    return data_dict

def plot_kv_diff_comparison(model_names, head_idx, output_path=None):
    """绘制KV Cache差异比较图"""
    
    # 创建子图
    fig, axes = plt.subplots(2, 2, figsize=(5, 3.5))
    # fig.suptitle(f'KV Cache Deviation', fontsize=14)
    
    # 定义颜色和样式
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
    
    for model_idx, model_name in enumerate(model_names):
        # 加载数据
        data = load_data(model_name, head_idx)
        
        # 提取重算比例和对应的差异比
        ratios = []
        key_diffs = []
        value_diffs = []
        
        for key, value in data.items():
            if key != 'none':  # 排除none情况
                ratios.append(key)
                key_diffs.append(value['key'])
                value_diffs.append(value['value'])
        
        # 转换为numpy数组并排序
        if ratios:
            ratios = np.array(ratios)
            key_diffs = np.array(key_diffs)
            value_diffs = np.array(value_diffs)
            
            # 按ratio排序
            sort_idx = np.argsort(ratios)
            ratios = ratios[sort_idx]
            key_diffs = key_diffs[sort_idx]
            value_diffs = value_diffs[sort_idx]
            
            # 计算均值和标准差
            key_mean = np.mean(key_diffs, axis=1)
            key_std = np.std(key_diffs, axis=1)
            value_mean = np.mean(value_diffs, axis=1)
            value_std = np.std(value_diffs, axis=1)
            
            # 左上：Key - Model 1
            if model_idx == 0:
                axes[0, 0].fill_between(ratios, key_mean - key_std, key_mean + key_std, 
                                      alpha=0.3, color=colors[0], label=' ± std')
                axes[0, 0].plot(ratios, key_mean, 'o-', color=colors[0], label='mean')
                axes[0, 0].set_title(f'Qwen3-4B-Instruct')
                # axes[0, 0].set_xlabel('Recompute Percentage')
                axes[0, 0].set_ylabel('Key Deviation')
                axes[0, 0].set_xlim(-5, 95)
                axes[0, 0].grid(True, alpha=0.3)
                axes[0, 0].legend()
            
            # 右上：Key - Model 2  
            elif model_idx == 1:
                axes[0, 1].fill_between(ratios, key_mean - key_std, key_mean + key_std, 
                                      alpha=0.3, color=colors[1], label=' ± std')
                axes[0, 1].plot(ratios, key_mean, 'o-', color=colors[1], label='mean')
                axes[0, 1].set_title(f'Llama3.1-8B-Instruct')
                # axes[0, 1].set_xlabel('Recompute Percentage')
                # axes[0, 1].set_ylabel('L2 Distance Ratio')
                axes[0, 1].set_xlim(-5, 95)
                axes[0, 1].grid(True, alpha=0.3)
                axes[0, 1].legend()
            
            # 左下：Value - Model 1
            if model_idx == 0:
                axes[1, 0].fill_between(ratios, value_mean - value_std, value_mean + value_std, 
                                      alpha=0.3, color=colors[0], label=' ± std')
                axes[1, 0].plot(ratios, value_mean, 'o-', color=colors[0], label='mean')
                # axes[1, 0].set_title(f'Value Differences - {model_names[0]}')
                axes[1, 0].set_xlabel('Recompute Percentage')
                axes[1, 0].set_ylabel('Value Deviation')
                axes[1, 0].set_xlim(-5, 95)
                axes[1, 0].grid(True, alpha=0.3)
                axes[1, 0].legend()
            
            # 右下：Value - Model 2
            elif model_idx == 1:
                axes[1, 1].fill_between(ratios, value_mean - value_std, value_mean + value_std, 
                                      alpha=0.3, color=colors[1], label=' ± std')
                axes[1, 1].plot(ratios, value_mean, 'o-', color=colors[1], label='mean')
                # axes[1, 1].set_title(f'Value Differences')
                axes[1, 1].set_xlabel('Recompute Percentage')
                # axes[1, 1].set_ylabel('L2 Distance Ratio')
                axes[1, 1].set_xlim(-5, 95)
                axes[1, 1].grid(True, alpha=0.3)
                axes[1, 1].legend()
    
    plt.tight_layout()
    plt.subplots_adjust(hspace=0.2, wspace=0.2)  # 减小间距
    
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
    
    plt.show()

# 使用示例
if __name__ == "__main__":
    # 绘制两个模型的对比图
    model_names = ["qwen3-4b", "llama3.1-8b"]  # 替换为实际的模型名称
    head_idx = 0  # 替换为需要分析的注意力头索引
    
    # 绘制双模型对比图
    plot_kv_diff_comparison(model_names, head_idx, "figures/kv_diff_comparison.png")
    
    # 或者绘制单个模型的图
    # plot_single_model_comparison("llama2-7b", head_idx, "kv_diff_single_model.png")
